# 2026-11-16 — Gen AI SDK practice

Run the setup cells top to bottom, then practise below.

## Setup

In [1]:
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig, Part, Tool, FunctionDeclaration

In [2]:
import google.auth
import os

# Detect the project ID from your gcloud login instead of hard-coding it
_, PROJECT_ID = google.auth.default()
PROJECT_ID = PROJECT_ID or os.environ.get("GOOGLE_CLOUD_PROJECT")
LOCATION = "global"
print("Project detected:", bool(PROJECT_ID))  # don't print the ID: repo is public

Project detected: True


In [3]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
MODEL_ID = "gemini-3.5-flash-lite"

In [4]:
# Cheap check that the client works (no text generated)
r = client.models.count_tokens(model=MODEL_ID, contents="test")
print(f"Client is ready! Token count: {r.total_tokens}")

Client is ready! Token count: 1


## Today's practice

In [5]:
! pip install langchain_community tiktoken langchain-openai langchain-google-genai langchain-classic langchain-text-splitters langchainhub chromadb langchain

In [6]:
import os
from dotenv import load_dotenv

# LangSmith tracing. The API key lives in the repo's .env file (gitignored), never in this notebook:
#   LANGCHAIN_API_KEY=lsv2_...
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
print("LangSmith key loaded:", bool(os.environ.get('LANGCHAIN_API_KEY')))

LangSmith key loaded: True


In [7]:
os.environ["USER_AGENT"] = "rag-learning"
EMBED_MODEL = "text-embedding-005"

In [ ]:
import bs4
# from langchain import hub                                   # old (removed in LangChain 1.x)
from langchain_classic import hub
# from langchain.text_splitter import RecursiveCharacterTextSplitter   # old (removed)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings   # OpenAI (original)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
# ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
# prompt = hub.pull("rlm/rag-prompt")   # old: deprecated, and public pulls are now blocked by default
from langchain_core.prompts import ChatPromptTemplate

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post_title", "post-header")
        )
    ),
)

docs = loader.load()
# print(docs)

text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
# print(splits)
print("\n\n".join(f"===== CHUNK {i} =====\n{d.page_content}" for i, d in enumerate(splits)))

# embed
vector_store = Chroma.from_documents(documents=splits,
                                     embedding=GoogleGenerativeAIEmbeddings(
                                         model=EMBED_MODEL,
                                         vertexai = True,
                                         project=PROJECT_ID,
                                         location=LOCATION
                                     ))

retriever = vector_store.as_retriever()

prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""
)

llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature = 0, vertexai = True, project = PROJECT_ID,
                             location = LOCATION)

# post processing

def format_docs(docs):
    return "/n/n".join(doc.page_content for doc in docs)

# chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")


In [ ]:
#  if you use just the retriever which is a wrapper around the vector store
retriever.invoke("What is Task Decomposition?")

In [ ]:
print(vector_store._collection.count())

In [ ]:
len(splits)

In [ ]:
# Write down the understand of what is happenning so far until the rag_chain.invoke



In [ ]:
# Part 2 : Indexing

In [ ]:
question = "What kind of pets do I like?"
document = "My favorite peet is a cat."

In [ ]:
# count the number of tokens in a string - this was from open ai
import tiktoken
def num_tokens_from_String(string: str, encoding_name: str) -> int:
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_String(question, "cl100k_base")


In [ ]:
# count the number of tokens used using gemini sdk
from google import genai
client = genai.Client(vertexai=True, project = PROJECT_ID, location=LOCATION)
result = client.models.count_tokens(model=MODEL_ID, contents=question)
print(result)

In [ ]:
# to know in detai9l how the words are grouped and converted into tokens
from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

resp = client.models.compute_tokens(model=MODEL_ID, contents=question)
info = resp.tokens_info[0]

print(len(info.token_ids), info.token_ids)
print([t.decode("utf-8", errors="replace") for t in info.tokens])

In [ ]:
# this is to understand how an embedding vector is created.
# In the embed query it takes a chunk from the document and then converts the chunk into a list of numbers called vector. To explain further the list of numbers are representation of a point in a n dimensional space. Lets say the embedding model can 
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embed = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, vertexai=True, project=PROJECT_ID, location=LOCATION)
query_result = embed.embed_query(question)
document_result = embed.embed_query(document)
print(len(query_result))
print(len(document_result))

In [ ]:
# find the cosine similarity between two enbedding vectors
import numpy as np
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product/(norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Consine similarity:", similarity)

In [8]:
# indexing
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs= dict(
        parse_only = bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

/var/folders/z_/9h2p27bj56n89wd3vmf52w600000gn/T/ipykernel_54993/1798065610.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [9]:
# split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

splits = text_splitter.split_documents(blog_docs)

In [10]:
print(splits)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refi

In [11]:
# use the split chunks into the embedding model to convert into the vectors

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorStore = Chroma.from_documents(documents=splits,
                                    embedding=GoogleGenerativeAIEmbeddings(
                                        model=EMBED_MODEL,
                                        vertexai=True,
                                        project=PROJECT_ID,
                                        location=LOCATION
                                    ))

retriever = vectorStore.as_retriever()

In [12]:
print(len(blog_docs), "document -> ", len(splits), "chunks")

1 document ->  50 chunks


In [13]:
print("count in store:", vectorStore._collection.count())

count in store: 50


In [ ]:
# So the above counts mean
# if one document is like a pdf and it is sent to made splits and every split is considered as a chunk.
# Now we have the chunk which is sent to the embedding model so each entry becomes a dictionary of id, text, vector and metadata
# So technically how many chunks is how many entries in the vector store
# From above len(splits) is same as vectorStore._collection.count()

In [ ]:
# Part 3 Retrieval

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorStore = Chroma.from_documents(documents=splits,
                                    embedding=GoogleGenerativeAIEmbeddings(
                                        model=EMBED_MODEL,
                                        vertexai=True,
                                        project=PROJECT_ID,
                                        location=LOCATION
                                    ))

retriever = vectorStore.as_retriever(search_kwargs={"k" : 1})

In [15]:
docs = retriever.invoke("What is task decomposition?")

In [16]:
len(docs)

1

In [ ]:
# Part 4 Generation

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

template = """Answert the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answert the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [18]:
llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, project=PROJECT_ID, location=LOCATION)

In [19]:
chain = prompt | llm

In [20]:
chain.invoke({"context": docs, "question": "What is Task Decomposition?"})

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'Based on the provided context, task decomposition is a process—enhanced by techniques like Chain of Thought (CoT)—where a complicated or hard task is broken down into smaller, simpler, manageable, and multiple steps (or subgoals). \n\nIt can be accomplished in three ways:\n1. By LLM with simple prompting (e.g., "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?").\n2. By using task-specific instructions (e.g., "Write a story outline." for writing a novel).\n3. With human inputs.', 'extras': {'signature': 'AY89a18pEAq6jBajVkxeNxGztIkE7Js3WWyb02TYINeluhkS52BuUuuLkcx5rd4uvELNBU/JyBtdmiShoGGqgvOfHdWHuBZ6O21my3SfbKzlH5iYkFAoPDboknVvqkPX4//4yW6/lcemiJduHcSgRZlo2w0Az3U='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a20248-d091-7305-bc53-aa9530e1d1c8-0', tool_calls=[], invalid_tool_calls=[], usage_metad

In [ ]:
result = chain.invoke({"context": docs, "question": "What is Task Decomposition?"})
print(result.content[0]["text"])

In [ ]:
chain.invoke({"context": "Cats are mammals.", "question": "What is Task Decomposition?"})

In [ ]:
# Part 5 Multi Query

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

template = """YOu are an AI language model assistant, Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. By generating multiple perspectives on the useer question, your goal is to help the user overcome some of the militations of the distance-based similarity search. Provide these alternative questions separated by newlines. Original question: {question}
"""
prompt_perspectives = ChatPromptTemplate.from_template(template)
# print(prompt_perspectives)

llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)

generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)



In [ ]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]


question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)
print(docs.text)

In [ ]:
from operator import itemgetter
# from langchain_openai import ChatOpenAI   # OpenAI (original)
from langchain_google_genai import ChatGoogleGenerativeAI   # Google (Vertex AI mode)
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatOpenAI(temperature=0)   # OpenAI (original)
llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

In [ ]:
# Key is to compare the token usage between single query retrieval and multi query retrieval

In [ ]:
# Part 6 Rag fusion

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """
You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI

llm_fusion = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)
# llm_fusion = ChatOpenAI(temperature=0)

generate_queries = (
    prompt_rag_fusion
    | llm_fusion
    | StrOutputParser()                                         
    | (lambda x: [q for q in x.split("\n") if q.strip()])  
)

In [ ]:
def reciprocal_rank_fusion(results: list[list], k = 60):
    fused_scores = {}
    doc_by_key = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            key = (doc.page_content, doc.metadata.get("source"))
            doc_by_key.setdefault(key, doc)
            fused_scores[key] = fused_scores.get(key, 0) + 1 / (rank + k)


    return [
        (doc_by_key[key], score)
        for key, score in sorted(fused_scores.items(), key = lambda x: x[1], reverse=True)
    ]

question = "What is the task decomposition of LLM agents?"
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

In [ ]:
for doc, score in docs[:5]:
    print(f"{score:.4f} | {doc.page_content[:90]!r}")

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# The fusion step returns (doc, score) tuples. Strip the scores and keep the text.
def fused_to_context(ranked):
    return "\n\n".join(doc.page_content for doc, score in ranked)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion | fused_to_context,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question": question})

In [ ]:
# Part 7 decomposition
# convert a question to N sub questions. REtrieve answers for one questions then feed as context to the next question and so on

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI

# LLM (one model, reused for the rest of Part 7)
llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)
# llm = ChatOpenAI(temperature=0)

# Chain
generate_queries_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])   # drop blank lines
)

question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question": question})

In [ ]:
questions

In [ ]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


q_a_pairs = ""
for q in questions:

    rag_chain = (
        {"context": itemgetter("question") | retriever | format_docs,   # format_docs added: clean text, not Document reprs
         "question": itemgetter("question"),
         "q_a_pairs": itemgetter("q_a_pairs")}
        | decomposition_prompt
        | llm
        | StrOutputParser())

    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair

In [ ]:
answer

In [ ]:
# Part 8 Step back as query translation

In [21]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]

# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

prompt_step_back = ChatPromptTemplate.from_messages(     
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI

llm_step_back = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)
# llm_step_back = ChatOpenAI(temperature=0)

generate_queries_step_back = prompt_step_back | llm_step_back | StrOutputParser()

question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'what is task decomposition in the context of LLM agents?'

In [27]:
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Response prompt  <-- this is where response_prompt is defined
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

{normal_context}
{step_back_context}

Original Question: {question}
Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

# Debug tap: prints the query going into the retriever, passes it on unchanged
def tap(label):
    def _t(q):
        print(f"[{label}] query -> {q!r}")
        return q
    return RunnableLambda(_t)

# Guard: if the step-back result is blank, fall back to the original question
def safe_step_back(x):
    sb = generate_queries_step_back.invoke(x).strip()
    return sb if sb else x["question"]

chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | tap("normal") | retriever | format_docs,
        "step_back_context": RunnableLambda(safe_step_back) | tap("step_back") | retriever | format_docs,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": question})

[normal] query -> 'What is task decomposition for LLM agents?'


~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[step_back] query -> 'what is task decomposition?'


~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, task decomposition is a planning component used for complicated tasks that involve many steps, allowing an agent to break down a hard task into smaller, simpler, and more manageable steps. \n\nTask decomposition can be performed in several ways:\n1. **By LLM with simple prompting**, such as using phrases like *"Steps for XYZ.\\n1."* or *"What are the subgoals for achieving XYZ?"*\n2. **By using task-specific instructions**, such as *"Write a story outline."* when writing a novel.\n3. **With human inputs.**\n\nFurthermore, standard prompting techniques and frameworks are often used to facilitate this process:\n* **Chain of Thought (CoT):** Instructs the model to "think step by step" to utilize more test-time computation, transforming big tasks into multiple manageable steps while providing insight into the model\'s thinking process.\n* **Tree of Thoughts (ToT):** Extends CoT by exploring multiple reasoning possibilities at each step. It decomposes the pro

In [24]:
sb = generate_queries_step_back.invoke({"question": question})
print(repr(sb))          # is it '' or whitespace?

raw = (prompt_step_back | llm_step_back).invoke({"question": question})
print(repr(raw.content))
print(raw.response_metadata)   # look for finish_reason
print(raw.usage_metadata)      # look at output_token_details / reasoning

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'what is task decomposition?'


~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'what is task decomposition?', 'extras': {'signature': 'AY89a180HtKXt17H7kmbT8aPUQRCMTi0prb7r0pdYyc/XgLiWNuQtkR6wQ2WdCIXF/Gaseryv9Y9uqwiHCw+yE9GLwOFri8QQGCiT/Ixx8FSQR4p+goGx0Did0mEK/g5IcDWxCnx3VgDR42kkfzlBQJeBdmLxk4='}}]
{'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}
{'input_tokens': 88, 'output_tokens': 5, 'total_tokens': 93, 'input_token_details': {'cache_read': 0}}


In [28]:
# Part 9 HyDE as query translation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate   # was: langchain.prompts (removed in 1.x)
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI

# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

llm_hyde = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)
# llm_hyde = ChatOpenAI(temperature=0)

generate_docs_for_retrieval = (
    prompt_hyde | llm_hyde | StrOutputParser()
)

question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question": question})

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'**Task Decomposition for LLM Agents: A Conceptual Framework**\n\nTask decomposition is a foundational operational paradigm in the deployment of Large Language Model (LLM) agents, designed to bridge the capability gap between monolithic model outputs and complex, multi-step problem-solving requirements. Formally defined as the recursive process of breaking down a complex, high-level user objective into a structured sequence of manageable, atomic sub-tasks, decomposition mitigates the limitations of context window saturation, catastrophic forgetting, and error accumulation inherent in direct end-to-end generation. This methodology typically leverages the generative and zero-shot reasoning capabilities of the underlying LLM—often operationalized via frameworks such as Chain-of-Thought (CoT), Tree of Thoughts (ToT), or explicit planning modules—to autonomously formulate execution graphs, API call sequences, or logical step-by-step agendas. By isolating discrete phases of an operation (e.g

In [30]:
# Guard: if the generated passage is blank, fall back to the original question
def safe_hyde(x):
    passage = generate_docs_for_retrieval.invoke(x).strip()
    return passage if passage else x["question"]

# Retrieve
retrieval_chain_hyde = RunnableLambda(safe_hyde) | retriever    # renamed so it doesn't overwrite your multi-query chain
retrieved_docs = retrieval_chain_hyde.invoke({"question": question})
retrieved_docs

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a

In [31]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context": format_docs(retrieved_docs), "question": question})   # format_docs: clean text, not Document reprs

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, task decomposition is a technique where a complicated, hard task is broken down into multiple smaller, simpler, and more manageable steps. This allows an agent to plan ahead and utilize more test-time computation. \n\nTask decomposition can be performed in the following ways:\n1. By the LLM itself using simple prompting (e.g., "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?").\n2. By using task-specific instructions (e.g., "Write a story outline" for writing a novel).\n3. With human inputs.'

In [ ]:
# Part 10 Routing 
# Two types -
# Logical Routing 
# Semantic Routing

In [32]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field        # was: langchain_core.pydantic_v1 (removed in newer langchain-core)
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI

# Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

# LLM with function call
router_llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)
# llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
structured_llm = router_llm.with_structured_output(RouteQuery)

# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt_route = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router
router = prompt_route | structured_llm

In [34]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [35]:
result

RouteQuery(datasource='python_docs')

In [36]:
result.datasource

'python_docs'

In [37]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        ### Logic here
        return "chain for js_docs"
    else:
        ### Logic here
        return "golang_docs"

from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

In [38]:
full_chain.invoke({"question": question})

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'chain for python_docs'

In [39]:
# Semantic Routing

In [41]:
import numpy as np

def cosine_similarity(X, Y):
    """Cosine similarity between every row of X and every row of Y."""
    X = np.array(X, dtype=float)
    Y = np.array(Y, dtype=float)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)    # scale each vector to length 1
    Y = Y / np.linalg.norm(Y, axis=1, keepdims=True)
    return X @ Y.T                                       # dot products = cosine similarities

In [42]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

# Embed prompts
route_embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, vertexai=True, project=PROJECT_ID, location=LOCATION)
# embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = route_embeddings.embed_documents(prompt_templates)

# Route question to prompt
def prompt_router(input):
    # Embed question
    query_embedding = route_embeddings.embed_query(input["query"])
    # Compute similarity
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    # Chosen prompt
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

semantic_chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | router_llm                      
    | StrOutputParser()
)

print(semantic_chain.invoke("What's a black hole"))

Using PHYSICS


~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Think of a black hole as a region in space where gravity has become so intensely strong that nothing—not even light—can escape its pull. 

It is formed when a massive star collapses in on itself at the end of its life, crushing a huge amount of matter into an infinitely small point. Because no light gets out, they are completely invisible, but we know they exist by observing how their immense gravity affects nearby stars and gas. 

In short: it's the ultimate cosmic trapdoor.


In [ ]:
# Part 11 Query Structuring

In [ ]:
pip install youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install pytube

Note: you may need to restart the kernel to use updated packages.


In [41]:
from langchain_community.document_loaders import YoutubeLoader

docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=pbAd8O1Lvm4", add_video_info=True
).load()

docs[0].metadata

HTTPError: HTTP Error 400: Bad Request

In [42]:
import datetime
from typing import Literal, Optional, Tuple
from pydantic import BaseModel, Field      # was: langchain_core.pydantic_v1

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video "
            "title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None,
        description="Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified.",
    )

    def pretty_print(self) -> None:
        # was: self.__fields__ (deprecated in pydantic v2)
        for name, field in type(self).model_fields.items():
            value = getattr(self, name)
            if value is not None and value != field.default:
                print(f"{name}: {value}")

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""

prompt_structured = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

llm_structured = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0, vertexai=True, project=PROJECT_ID, location=LOCATION)
# llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

structured_llm = llm_structured.with_structured_output(TutorialSearch)
query_analyzer = prompt_structured | structured_llm

In [45]:
query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content_search: rag from scratch
title_search: rag from scratch


In [46]:
query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
).pretty_print()

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content_search: chat langchain
title_search: chat langchain
earliest_publish_date: 2023-01-01
latest_publish_date: 2024-01-01


In [47]:
query_analyzer.invoke(
    {"question": "videos that are focused on the topic of chat langchain that are published before 2024"}
).pretty_print()

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content_search: chat langchain
title_search: chat langchain
latest_publish_date: 2024-01-01


In [48]:
query_analyzer.invoke(
    {"question": "how to use multi-modal models in an agent, only videos under 5 minutes"}
).pretty_print()

~/code/ai-learning-in-gcp/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content_search: how to use multi-modal models in an agent
title_search: multi-modal models agent
max_length_sec: 300
